# **06 - Dashboard Export**

This notebook builds the three JSON files utilized in the HTML dashboard, starting from the LSTM's test-set predictions and joining in blocker name, position, team, and week from the player, play and game context csvs. Before aggregating, the raw predicted probabilities were checked against the actual win rate and found to be substantially miscalibrated as a side effect of the recall-biased class weighting used in training; Platt scaling was applied to correct this, so downstream expected-win-rate and win-rate-over-expected figures reflect genuine performance rather than an artifact of that weighting. Blocker-level, team-level (snap-weighted), and matchup-level tables were then built and exported as JSON.

In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.linear_model import LogisticRegression

lstm_preds = pd.read_parquet("../processed/lstm_predictions.parquet")
players = pd.read_csv("../nfl-big-data-bowl-2023/players.csv")
plays = pd.read_csv("../nfl-big-data-bowl-2023/plays.csv")
games = pd.read_csv("../nfl-big-data-bowl-2023/games.csv")

lstm_preds.tail()

In [ ]:
lstm_preds = lstm_preds.merge(
    players[["nflId", "displayName", "officialPosition"]].rename(
        columns={"displayName": "blockerName", "officialPosition": "blockerPosition"}
    ),
    on="nflId", how="left",
)

lstm_preds = lstm_preds.merge(
    players[["nflId", "displayName"]].rename(
        columns={"nflId": "rusher_nflId", "displayName": "rusherName"}
    ),
    on="rusher_nflId", how="left",
)

lstm_preds = lstm_preds.merge(
    plays[["gameId", "playId", "possessionTeam", "defensiveTeam"]],
    on=["gameId", "playId"], how="left",
)
lstm_preds = lstm_preds.rename(columns={"possessionTeam": "team"})

lstm_preds = lstm_preds.merge(
    games[["gameId", "week"]],
    on="gameId", how="left",
)

lstm_preds.tail()

In [ ]:
overall_actual_win_rate = 1 - lstm_preds["blocker_lost_actual"].mean()
overall_expected_win_rate = lstm_preds["blocker_win_probability"].mean()

print(f"Actual win rate (whole test set): {overall_actual_win_rate:.4f}")
print(f"Model's average predicted win probability: {overall_expected_win_rate:.4f}")

raw_proba = lstm_preds["blocker_lost_predicted_proba"].values.reshape(-1, 1)
true_label = lstm_preds["blocker_lost_actual"].values

calibrator = LogisticRegression()
calibrator.fit(raw_proba, true_label)

lstm_preds["blocker_lost_calibrated"] = calibrator.predict_proba(raw_proba)[:, 1]
lstm_preds["blocker_win_probability"] = 1 - lstm_preds["blocker_lost_calibrated"]

calibrated_expected = lstm_preds["blocker_win_probability"].mean()
print(f"Calibrated predicted win rate: {calibrated_expected:.4f}")

In [ ]:
snapshot_features = pd.read_parquet("../processed/snapshot_features.parquet")
print(f"Full snapshot_features: {len(snapshot_features):,} rows")

%run 01_ingest.ipynb

blockers = tracking_df[tracking_df["pff_role"] == "Pass Block"].copy()

rusher_cols = ["gameId", "playId", "frameId", "nflId", "x", "y", "s", "a", "o", "dir"]
rushers = tracking_df[rusher_cols].rename(columns={
    "nflId": "rusher_nflId",
    "x": "rusher_x", "y": "rusher_y",
    "s": "rusher_s", "a": "rusher_a",
    "o": "rusher_o", "dir": "rusher_dir",
})

paired = blockers.merge(
    rushers,
    left_on=["gameId", "playId", "frameId", "pff_nflIdBlockedPlayer"],
    right_on=["gameId", "playId", "frameId", "rusher_nflId"],
    how="inner",
)

paired["distance"] = np.sqrt(
    (paired["x"] - paired["rusher_x"])**2 + (paired["y"] - paired["rusher_y"])**2
)
paired = paired.sort_values(["gameId", "playId", "nflId", "frameId"])
paired["prev_distance"] = paired.groupby(["gameId", "playId", "nflId"])["distance"].shift(1)
paired["closing_speed"] = (paired["prev_distance"] - paired["distance"]).fillna(0)
paired["orientation_diff"] = (paired["o"] - paired["rusher_o"]).abs()
paired["dx"] = paired["x"] - paired["rusher_x"]
paired["dy"] = paired["y"] - paired["rusher_y"]
paired["speed_diff"] = paired["s"] - paired["rusher_s"]

seq_feature_cols = ["dx", "dy", "distance", "closing_speed", "orientation_diff", "speed_diff", "s", "a"]

sequences = []
keys = []
for key, group in paired.groupby(["gameId", "playId", "nflId", "rusher_nflId"]):
    group = group.sort_values("frameId")
    sequences.append(group[seq_feature_cols].values.astype("float32"))
    keys.append(key)

MAX_LEN = 54
n_features = len(seq_feature_cols)

def pad_sequence(seq, max_len):
    length = min(len(seq), max_len)
    padded = np.zeros((max_len, n_features), dtype="float32")
    padded[:length] = seq[:length]
    return padded, length

padded_seqs, true_lengths = [], []
for seq in sequences:
    padded, length = pad_sequence(seq, MAX_LEN)
    padded_seqs.append(padded)
    true_lengths.append(length)

X_full = np.stack(padded_seqs)
full_lengths = np.array(true_lengths)
full_keys_df = pd.DataFrame(keys, columns=["gameId", "playId", "nflId", "rusher_nflId"])

print(f"Full sequence set: {X_full.shape[0]:,} matchups")

In [ ]:
import torch
import torch.nn as nn

class BlockOutcomeLSTM(nn.Module):
    def __init__(self, n_features, hidden_size=32):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden_size, batch_first=True)
        self.fc1 = nn.Linear(hidden_size, 16)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(16, 1)

    def forward(self, x, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            x, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        _, (hidden, _) = self.lstm(packed)
        out = self.relu(self.fc1(hidden[-1]))
        out = self.fc2(out)
        return out.squeeze(1)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")

model = BlockOutcomeLSTM(n_features=n_features).to(device)
model.load_state_dict(torch.load("../models/lstm_sequence.pt", map_location=device))
model.eval()

def get_predictions(model, X, lengths, device, batch_size=64):
    probs = []
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            X_batch = torch.tensor(X[i:i+batch_size], dtype=torch.float32).to(device)
            len_batch = torch.tensor(lengths[i:i+batch_size], dtype=torch.int64)
            logits = model(X_batch, len_batch)
            probs.extend(torch.sigmoid(logits).cpu().numpy())
    return np.array(probs)

full_raw_proba = get_predictions(model, X_full, full_lengths, device)
full_calibrated_proba = calibrator.predict_proba(full_raw_proba.reshape(-1, 1))[:, 1]

full_keys_df["blocker_lost_predicted_proba"] = full_raw_proba
full_keys_df["blocker_lost_calibrated"] = full_calibrated_proba
full_keys_df["blocker_win_probability"] = 1 - full_calibrated_proba

print(f"Inference complete on {len(full_keys_df):,} matchups")
full_keys_df.head()

In [ ]:
outcome_cols = ["gameId", "playId", "nflId", "rusher_nflId", "pff_sackAllowed",
                "pff_hurryAllowed", "pff_hitAllowed", "pff_beatenByDefender"]

full_preds = full_keys_df.merge(
    snapshot_features[outcome_cols], on=["gameId", "playId", "nflId", "rusher_nflId"], how="left"
)

full_preds["blocker_lost_actual"] = (
    full_preds["pff_sackAllowed"].astype(bool)
    | full_preds["pff_hurryAllowed"].astype(bool)
    | full_preds["pff_hitAllowed"].astype(bool)
    | full_preds["pff_beatenByDefender"].astype(bool)
).astype(int)

full_preds = full_preds.merge(
    players[["nflId", "displayName", "officialPosition"]].rename(
        columns={"displayName": "blockerName", "officialPosition": "blockerPosition"}
    ),
    on="nflId", how="left",
)
full_preds = full_preds.merge(
    players[["nflId", "displayName"]].rename(
        columns={"nflId": "rusher_nflId", "displayName": "rusherName"}
    ),
    on="rusher_nflId", how="left",
)
full_preds = full_preds.merge(
    plays[["gameId", "playId", "possessionTeam", "defensiveTeam"]],
    on=["gameId", "playId"], how="left",
)
full_preds = full_preds.rename(columns={"possessionTeam": "team"})
full_preds = full_preds.merge(games[["gameId", "week"]], on="gameId", how="left")

print(f"Full joined dataset: {len(full_preds):,} rows")
print(full_preds[["blockerName", "blockerPosition", "rusherName", "team", "week"]].isna().sum())
full_preds.head()

In [ ]:
blocker_agg = (
    full_preds
    .groupby(["nflId", "blockerName", "blockerPosition", "team"])
    .agg(
        snaps=("blocker_lost_actual", "count"),
        wins=("blocker_lost_actual", lambda x: (x == 0).sum()),
        losses=("blocker_lost_actual", lambda x: (x == 1).sum()),
        expectedWinRate=("blocker_win_probability", "mean"),
    )
    .reset_index()
)

blocker_agg["winRate"] = blocker_agg["wins"] / blocker_agg["snaps"]
blocker_agg["winRateOverExpected"] = blocker_agg["winRate"] - blocker_agg["expectedWinRate"]

MIN_SNAPS = 80
OL_POSITIONS = {"C", "G", "T"}

before_count = len(blocker_agg)
blocker_agg = blocker_agg[
    (blocker_agg["snaps"] >= MIN_SNAPS) &
    (blocker_agg["blockerPosition"].isin(OL_POSITIONS))
].reset_index(drop=True)

print(f"Blockers before filtering: {before_count:,}")
print(f"Blockers after filtering (>= {MIN_SNAPS} snaps, OL positions only): {len(blocker_agg):,}")
blocker_agg.head()

In [ ]:
nflverse_players = pd.read_csv(
    "https://github.com/nflverse/nflverse-data/releases/download/players/players.csv"
)

print(f"nflverse players file: {len(nflverse_players):,} rows")
print([c for c in nflverse_players.columns if "name" in c.lower() or "headshot" in c.lower()])

In [ ]:
name_overrides = {
    "J.C. Tretter": "JC Tretter",
    "Charles Leno": "Charles Leno Jr.",
    "Shaquille Mason": "Shaq Mason",
    "Julien Davenport": "Julie'n Davenport",
    "Yosuah Nijman": "Yosh Nijman",
    "Lloyd Cushenberry": "Lloyd Cushenberry III",
    "Michael Onwenu": "Mike Onwenu",
    "Samuel Cosmi": "Sam Cosmi",
    "Michael Jordan": "Mike Jordan",
}

blocker_agg["nameForLookup"] = blocker_agg["blockerName"].replace(name_overrides)

headshot_lookup = nflverse_players[["display_name", "headshot"]].drop_duplicates(subset="display_name")

blocker_agg = blocker_agg.merge(
    headshot_lookup, left_on="nameForLookup", right_on="display_name", how="left"
)
blocker_agg = blocker_agg.drop(columns=["display_name", "nameForLookup"])
blocker_agg = blocker_agg.rename(columns={"headshot": "headshotUrl"})

In [ ]:
def snap_weighted_mean(group):
    return np.average(group["expectedWinRate"], weights=group["snaps"])

team_agg = (
    blocker_agg
    .groupby("team")
    .apply(lambda g: pd.Series({
        "blockers": g["nflId"].nunique(),
        "totalSnaps": g["snaps"].sum(),
        "totalWins": g["wins"].sum(),
        "expectedWinRate": snap_weighted_mean(g),
    }), include_groups=False)
    .reset_index()
)

team_agg["winRate"] = team_agg["totalWins"] / team_agg["totalSnaps"]
team_agg["winRateOverExpected"] = team_agg["winRate"] - team_agg["expectedWinRate"]

team_agg.tail()

In [ ]:
division_map = {
    "BUF": "AFC East", "MIA": "AFC East", "NE": "AFC East", "NYJ": "AFC East",
    "BAL": "AFC North", "CIN": "AFC North", "CLE": "AFC North", "PIT": "AFC North",
    "HOU": "AFC South", "IND": "AFC South", "JAX": "AFC South", "TEN": "AFC South",
    "DEN": "AFC West", "KC": "AFC West", "LV": "AFC West", "LAC": "AFC West",
    "DAL": "NFC East", "NYG": "NFC East", "PHI": "NFC East", "WAS": "NFC East",
    "CHI": "NFC North", "DET": "NFC North", "GB": "NFC North", "MIN": "NFC North",
    "ATL": "NFC South", "CAR": "NFC South", "NO": "NFC South", "TB": "NFC South",
    "ARI": "NFC West", "LA": "NFC West", "SF": "NFC West", "SEA": "NFC West",
}

team_agg["division"] = team_agg["team"].map(division_map)

team_agg.head()

In [ ]:
matchup_export = (
    full_preds
    .groupby(["blockerName", "rusherName", "team", "defensiveTeam", "week"])
    .agg(
        snaps=("blocker_lost_actual", "count"),
        wins=("blocker_lost_actual", lambda x: (x == 0).sum()),
        losses=("blocker_lost_actual", lambda x: (x == 1).sum()),
        expectedWinRate=("blocker_win_probability", "mean"),
    )
    .reset_index()
)

matchup_export = matchup_export.rename(columns={
    "blockerName": "blocker",
    "rusherName": "rusher",
    "team": "blockerTeam",
    "defensiveTeam": "team",
})
matchup_export["winRate"] = matchup_export["wins"] / matchup_export["snaps"]
matchup_export["winRateOverExpected"] = matchup_export["winRate"] - matchup_export["expectedWinRate"]

matchup_export.head()

In [ ]:
blocker_output = blocker_agg.to_dict(orient="records")
team_output = team_agg.to_dict(orient="records")
matchup_output = matchup_export.to_dict(orient="records")

with open("../dashboard/data/dashboard_data.json", "w") as f:
    json.dump(blocker_output, f, indent=2)

with open("../dashboard/data/team_data.json", "w") as f:
    json.dump(team_output, f, indent=2)

with open("../dashboard/data/matchup_data.json", "w") as f:
    json.dump(matchup_output, f, indent=2)

print("All three JSON files written to ../dashboard/data/")

In [ ]:
full_preds.to_parquet("../processed/full_preds.parquet", index=False)
print(f"Saved full_preds ({len(full_preds):,} rows)")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

full_preds = pd.read_parquet("../processed/full_preds.parquet")

snaps_before_filter = (
    full_preds
    .groupby(["nflId", "blockerName", "blockerPosition", "team"])
    .agg(snaps=("blocker_lost_actual", "count"))
    .reset_index()
)

print(f"Blockers before filtering: {len(snaps_before_filter):,}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(snaps_before_filter["snaps"], bins=20, color="#a6a396", edgecolor="#4d4b43")
ax.axvline(80, color="#c98a2c", linestyle="--", linewidth=2, label="Chosen cutoff (80 snaps)")
ax.set_xlabel("Season Snap Count")
ax.set_ylabel("Number of Blockers")
ax.set_title("Full-Season Snap-Count Distribution")
ax.legend()
plt.tight_layout()
plt.savefig("../figures/snap_distribution.png", dpi=150, bbox_inches="tight")